# Ecuación de transporte 1D: cuaderno de solución


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def u0(x):
    return 0.90 * np.exp(-((x - 0.30) ** 2) / 0.0025) + 0.35 * np.exp(-((x - 0.75) ** 2) / 0.008)


def exact_solution(x, t, a=1.0, L=1.0):
    return u0((x - a * t) % L)


def build_grid(L=1.0, T=1.4, N=320, a=1.0, nu_obj=0.98):
    dx = L / N
    dt = nu_obj * dx / a
    nt = int(np.ceil(T / dt))
    dt = T / nt
    nu = a * dt / dx
    x = np.linspace(0.0, L, N, endpoint=False)
    return x, dx, dt, nt, nu


def upwind_step(u, nu):
    return u - nu * (u - np.roll(u, 1))


def solve_transport(a=1.0, L=1.0, T=1.4, N=320, nu_obj=0.98, store_every=5):
    x, dx, dt, nt, nu = build_grid(L=L, T=T, N=N, a=a, nu_obj=nu_obj)
    u = u0(x).copy()
    t_hist = [0.0]
    U_hist = [u.copy()]
    for n in range(nt):
        u = upwind_step(u, nu)
        t_next = (n + 1) * dt
        if (n + 1) % store_every == 0 or (n + 1) == nt:
            t_hist.append(t_next)
            U_hist.append(u.copy())
    return x, u, nt * dt, nu, np.array(t_hist), np.array(U_hist)


x, u_num, t_final, nu, t_hist, U_hist = solve_transport()
u_ex = exact_solution(x, t_final)
err_l2 = np.sqrt(np.mean((u_num - u_ex) ** 2))
print(f"t_final={t_final:.6f}")
print(f"nu={nu:.6f}")
print(f"error L2={err_l2:.3e}")


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(x, u_num, lw=2.2, label="Numérica")
plt.plot(x, u_ex, "--", lw=2.0, label="Exacta")
plt.title("Transporte 1D en t_final")
plt.xlabel("x")
plt.ylabel("u(x,t_final)")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()
